# CIFAR-10 Deep Learning Classifier — Analysis Notebook

**Author:** Md. Mehenuf Hossain Bhuiyan  
**Repository:** [github.com/mehenuf/cifar10-classifier](https://github.com/mehenuf/cifar10-classifier)  
**Model:** Custom ResNet-style CNN + SE Attention | **Test Accuracy:** 94.81%+ | **Hardware:** NVIDIA RTX 5070 (CUDA)

---

## Notebook Overview

| Section | Contents |
|---------|----------|
| **1. Environment Setup** | Imports, device detection, reproducibility |
| **2. Dataset Exploration** | Class distribution, sample grid, pixel statistics |
| **3. Augmentation Visualisation** | Raw vs augmented + CutMix examples |
| **4. Model Architecture** | SE-ResNet layer summary, parameter count |
| **5. Training History** | Loss curves, accuracy curves, LR schedule |
| **6. Test Set Evaluation** | Full metrics from evaluation_report.json |
| **7. Confusion Matrix** | Heatmap + worst confusion pairs |
| **8. Per-Class Deep Dive** | Precision / Recall / F1 / Accuracy charts |
| **9. Calibration Analysis** | ECE before/after temperature scaling |
| **10. Error Analysis** | Top confident mistakes, cat/dog & auto/truck deep dive |
| **11. Improvement Analysis** | SE Attention + CutMix + Focal Loss impact |
| **12. Live Inference Demo** | Run model on custom images inside notebook |
| **13. Summary & Conclusions** | Key findings, improvements, future directions |

---
## 1. Environment Setup

In [ ]:
%matplotlib inline

# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# ── Resolve project root ───────────────────────────────────────────────────────
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.makedirs('./results', exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

# ── Numerical & ML ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Project modules ───────────────────────────────────────────────────────────
from src.data_utils import (
    get_dataloaders, CLASSES, CIFAR10_MEAN, CIFAR10_STD,
    denormalise, get_train_transforms, cutmix_batch
)
from src.model   import build_model
from src.utilities   import get_device, load_checkpoint, set_seed, FocalLoss
from src.predict import CIFAR10Predictor

# ── Global plot style — dark theme ────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#30363D',
    'axes.labelcolor':  '#C9D1D9',
    'xtick.color':      '#8B949E',
    'ytick.color':      '#8B949E',
    'text.color':       '#C9D1D9',
    'grid.color':       '#21262D',
    'legend.facecolor': '#161B22',
    'legend.edgecolor': '#30363D',
    'figure.dpi':       110,
})
PALETTE = ['#60A5FA','#F87171','#34D399','#FBBF24','#A78BFA',
           '#F472B6','#2DD4BF','#FB923C','#818CF8','#86EFAC']

# ── Seed & device ─────────────────────────────────────────────────────────────
set_seed(42)
DEVICE = get_device()

print(f'\n✅ Environment ready')
print(f'   PyTorch  : {torch.__version__}')
print(f'   Device   : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU      : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'   Classes  : {CLASSES}')

Project root: z:\Project\cifar10-classifier


ModuleNotFoundError: No module named 'src.utils'

---
## 2. Dataset Exploration

CIFAR-10 contains **60,000 images** (32×32 px, RGB) across **10 balanced classes** with 6,000 images each.

In [ ]:
# Load data
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir='./data', batch_size=64, augment=False
)
print(f'Train : {len(train_loader.dataset):,}')
print(f'Val   : {len(val_loader.dataset):,}')
print(f'Test  : {len(test_loader.dataset):,}')

In [ ]:
# ── 2a. Sample grid — 8 images per class ──────────────────────────────────────
raw_ds = datasets.CIFAR10('./data', train=True, download=False,
                           transform=transforms.ToTensor())

N_PER = 8
class_images = {cls: [] for cls in CLASSES}
for img, lbl in raw_ds:
    cls = CLASSES[lbl]
    if len(class_images[cls]) < N_PER:
        class_images[cls].append(img)
    if all(len(v) == N_PER for v in class_images.values()):
        break

EMOJIS = ['✈️','🚗','🐦','🐱','🦌','🐶','🐸','🐴','🚢','🚚']
fig, axes = plt.subplots(10, N_PER, figsize=(N_PER*1.4, 10*1.4))
fig.suptitle('CIFAR-10 Dataset — 8 Samples per Class', fontsize=14, y=1.01)

for r, cls in enumerate(CLASSES):
    for c, img in enumerate(class_images[cls]):
        axes[r, c].imshow(img.permute(1,2,0).numpy())
        axes[r, c].axis('off')
    axes[r, 0].set_ylabel(f'{EMOJIS[r]} {cls}', fontsize=9,
                           color=PALETTE[r], rotation=0,
                           labelpad=55, va='center')
plt.tight_layout()
plt.savefig('./results/nb_sample_grid.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2b. Class distribution ────────────────────────────────────────────────────
train_labels = [lbl for _, lbl in raw_ds]
counts = [train_labels.count(i) for i in range(10)]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(CLASSES, counts, color=PALETTE, edgecolor='#21262D')
ax.bar_label(bars, fmt='%d', fontsize=9, padding=4)
ax.set_ylim(0, 6500)
ax.set_title('Training Set Class Distribution — Perfectly Balanced', fontsize=13)
ax.set_ylabel('Number of images')
ax.axhline(5000, color='#F87171', ls='--', lw=1.2, label='Expected (5,000)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print('Each class has exactly 5,000 training images — no class imbalance.')

In [ ]:
# ── 2c. Pixel intensity distribution per channel ──────────────────────────────
sample_batch, _ = next(iter(train_loader))
channel_names  = ['Red', 'Green', 'Blue']
channel_colors = ['#F87171', '#34D399', '#60A5FA']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Pixel Intensity Distribution per Channel (after normalisation)', fontsize=12)

for i, (name, color) in enumerate(zip(channel_names, channel_colors)):
    data = sample_batch[:, i].flatten().numpy()
    axes[i].hist(data, bins=60, color=color, alpha=0.8, edgecolor='none')
    axes[i].axvline(data.mean(), color='white', ls='--', lw=1.5,
                    label=f'Mean: {data.mean():.3f}')
    axes[i].set_title(name, fontsize=11, color=color)
    axes[i].set_xlabel('Normalised pixel value')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=9); axes[i].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Normalisation: mean={CIFAR10_MEAN}, std={CIFAR10_STD}')

---
## 3. Augmentation Visualisation

Three augmentation strategies are now in use: the original pipeline, plus **CutMix** (new) which specifically targets cat/dog and automobile/truck confusion.

In [ ]:
# ── 3a. Original augmentation pipeline — raw vs augmented ─────────────────────
aug_ds = datasets.CIFAR10('./data', train=True, download=False,
                           transform=get_train_transforms(augment=True))
mean_t = torch.tensor(CIFAR10_MEAN).view(3,1,1)
std_t  = torch.tensor(CIFAR10_STD).view(3,1,1)

N = 8
fig, axes = plt.subplots(2, N, figsize=(N*1.8, 5))
fig.suptitle('Standard Augmentation — Raw (top) vs Augmented (bottom)', fontsize=12, y=1.02)

for col in range(N):
    raw_img, lbl = raw_ds[col * 500]
    aug_img, _   = aug_ds[col * 500]
    axes[0, col].imshow(raw_img.permute(1,2,0).numpy())
    axes[0, col].set_title(CLASSES[lbl], fontsize=8, color=PALETTE[lbl])
    axes[0, col].axis('off')
    aug_vis = (aug_img * std_t + mean_t).clamp(0,1)
    axes[1, col].imshow(aug_vis.permute(1,2,0).numpy())
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10, rotation=0, labelpad=50, va='center')
axes[1, 0].set_ylabel('Augmented', fontsize=10, rotation=0, labelpad=55, va='center')
plt.tight_layout(); plt.show()

print('Standard pipeline: RandomCrop → HFlip → ColorJitter → Normalise → RandomErasing')

In [ ]:
# ── 3b. CutMix visualisation ──────────────────────────────────────────────────
# CutMix is the key new augmentation targeting cat/dog and auto/truck confusion.
# It cuts a patch from one image and pastes it into another, mixing labels
# proportionally. This forces the model to learn partial object features.

aug_loader = get_dataloaders('./data', batch_size=8, augment=True)[0]
imgs, lbls = next(iter(aug_loader))

mixed_imgs, labels_a, labels_b, lam = cutmix_batch(imgs, lbls, alpha=1.0)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle(f'CutMix Augmentation — λ={lam:.2f} (NEW: targets cat/dog & auto/truck confusion)',
             fontsize=11, y=1.02)

for col in range(8):
    orig = denormalise(imgs[col]).permute(1,2,0).numpy()
    mix  = denormalise(mixed_imgs[col]).permute(1,2,0).numpy()
    axes[0, col].imshow(orig)
    axes[0, col].set_title(CLASSES[lbls[col].item()], fontsize=8, color=PALETTE[lbls[col].item()])
    axes[0, col].axis('off')
    axes[1, col].imshow(mix.clip(0,1))
    a_cls = CLASSES[labels_a[col].item()]
    b_cls = CLASSES[labels_b[col].item()]
    axes[1, col].set_title(f'{a_cls}+{b_cls}', fontsize=7)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=9, rotation=0, labelpad=50, va='center')
axes[1, 0].set_ylabel('CutMix', fontsize=9, color='#34D399', rotation=0, labelpad=45, va='center')
plt.tight_layout()
plt.savefig('./results/nb_cutmix_examples.png', dpi=110, bbox_inches='tight')
plt.show()

print(f'CutMix λ={lam:.3f} means {lam*100:.1f}% of the loss comes from the original label')
print('and', f'{(1-lam)*100:.1f}% from the pasted patch label.')
print('This forces the model to recognise objects from partial views — reducing inter-class confusion.')

---
## 4. Model Architecture — SE-ResNet

The updated model adds **Squeeze-and-Excitation (SE) channel attention** inside every residual block. SE blocks learn to reweight feature channels — giving more importance to channels that are discriminative for the current image. This directly targets cat/dog and automobile/truck confusion.

In [ ]:
# ── 4a. Build model and print summary ─────────────────────────────────────────
model = build_model(num_classes=10, dropout=0.3).to(DEVICE)

print('=' * 65)
print('  CIFAR10Net (SE-ResNet) — Architecture Summary')
print('=' * 65)
total_params = 0
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    total_params += n
    print(f'  {name:<14} {module.__class__.__name__:<22} {n:>12,} params')
print('-' * 65)
print(f'  {"TOTAL":<36} {total_params:>12,} ({total_params/1e6:.2f}M)')
print('=' * 65)
print()
print('New in this version:')
print('  ✅ SEBlock inside every ResidualBlock (channel attention)')
print('  ✅ CutMix augmentation during training')
print('  ✅ Focal Loss (γ=2.0) replacing CrossEntropy')
print('  ✅ Temperature scaling post-training')

In [ ]:
# ── 4b. SE Block explanation ──────────────────────────────────────────────────
print('How SE (Squeeze-and-Excitation) Attention Works:')
print()
print('  Standard ResBlock:  Input → Conv → BN → Conv → BN → + Skip → GELU')
print('  SE-ResBlock:        Input → Conv → BN → Conv → BN → SE → + Skip → GELU')
print()
print('  SE Block internals:')
print('    1. Global Average Pool  → squeeze spatial dims to (B, C)')
print('    2. Linear(C → C//16)    → compress to bottleneck')
print('    3. ReLU')
print('    4. Linear(C//16 → C)    → expand back')
print('    5. Sigmoid              → scale each channel 0–1')
print('    6. Multiply             → reweight original feature map channels')
print()
print('  Effect: channels containing cat-specific features (whiskers, ears)')
print('  get amplified; dog-like features get suppressed for cat images.')
print('  This is exactly what reduces cat/dog confusion.')

In [ ]:
# ── 4c. Parameter distribution ────────────────────────────────────────────────
stage_params = {}
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    stage_params[name] = n

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
names  = list(stage_params.keys())
values = list(stage_params.values())

bars = ax1.bar(names, [v/1e6 for v in values], color=PALETTE[:len(names)], edgecolor='#21262D')
ax1.bar_label(bars, fmt='%.2fM', fontsize=9, padding=4)
ax1.set_title('Parameters per Module (Millions)', fontsize=11)
ax1.set_ylabel('Parameters (M)'); ax1.grid(axis='y', alpha=0.3)

ax2.pie(values, labels=names, colors=PALETTE[:len(names)],
        autopct='%1.1f%%', startangle=140,
        textprops={'color': '#C9D1D9', 'fontsize': 9})
ax2.set_title('Parameter Distribution', fontsize=11)

fig.suptitle(f'SE-CIFAR10Net — {total_params/1e6:.2f}M Total Parameters', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4d. Forward pass shape trace ──────────────────────────────────────────────
model.eval()
x = torch.randn(1, 3, 32, 32).to(DEVICE)
print('Forward pass — spatial dimensions at each stage:')
with torch.no_grad():
    print(f'  Input         : {tuple(x.shape)}')
    x = model.stem(x);   print(f'  After stem    : {tuple(x.shape)}')
    x = model.stage1(x); print(f'  After stage1  : {tuple(x.shape)}')
    x = model.stage2(x); print(f'  After stage2  : {tuple(x.shape)}')
    x = model.stage3(x); print(f'  After stage3  : {tuple(x.shape)}')
    x = model.stage4(x); print(f'  After stage4  : {tuple(x.shape)}')
    x = model.gap(x).flatten(1); print(f'  After GAP     : {tuple(x.shape)}')
    x = model.classifier(x);     print(f'  Output logits : {tuple(x.shape)}')

---
## 5. Training History

In [ ]:
# ── 5a. Load history ──────────────────────────────────────────────────────────
with open('./results/training_history.json') as f:
    history = json.load(f)

epochs     = range(1, len(history['train_loss']) + 1)
best_epoch = int(np.argmax(history['val_top1'])) + 1
best_val   = max(history['val_top1']) * 100

print(f'Epochs trained    : {history["epochs_trained"]}')
print(f'Best val Top-1    : {best_val:.2f}% at epoch {best_epoch}')
print(f'Total train time  : {history["total_time_sec"]/60:.1f} minutes')

In [ ]:
# ── 5b. Training curves ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(17, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.32)

ax0 = fig.add_subplot(gs[0])
ax0.plot(epochs, history['train_loss'], color='#60A5FA', lw=1.5, label='Train')
ax0.plot(epochs, history['val_loss'],   color='#F87171', lw=1.5, label='Val')
ax0.axvline(best_epoch, color='#34D399', ls=':', lw=1.2, label=f'Best ({best_epoch})')
ax0.set_title('Loss (Focal Loss)', fontsize=11)
ax0.set_xlabel('Epoch'); ax0.set_ylabel('Loss')
ax0.legend(fontsize=8); ax0.grid(alpha=0.3)

ax1 = fig.add_subplot(gs[1])
ax1.plot(epochs, [v*100 for v in history['train_top1']],
         color='#60A5FA', lw=1.5, label='Train Top-1')
ax1.plot(epochs, [v*100 for v in history['val_top1']],
         color='#F87171', lw=1.5, label='Val Top-1')
ax1.plot(epochs, [v*100 for v in history['train_top5']],
         color='#60A5FA', lw=1, ls='--', alpha=0.5, label='Train Top-5')
ax1.plot(epochs, [v*100 for v in history['val_top5']],
         color='#F87171', lw=1, ls='--', alpha=0.5, label='Val Top-5')
ax1.axvline(best_epoch, color='#34D399', ls=':', lw=1.2)
ax1.axhline(best_val, color='#34D399', ls=':', lw=1.0, label=f'Best {best_val:.2f}%')
ax1.set_title('Top-1 & Top-5 Accuracy', fontsize=11)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy (%)')
ax1.legend(fontsize=7); ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[2])
ax2.plot(epochs, history['lr'], color='#34D399', lw=1.5)
ax2.set_title('LR — CosineAnnealing', fontsize=11)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Learning Rate')
ax2.set_yscale('log'); ax2.grid(alpha=0.3)

fig.suptitle(f'Training History — Best Val Top-1: {best_val:.2f}% @ Epoch {best_epoch}',
             fontsize=12, y=1.02)
plt.savefig('./results/nb_training_curves.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5c. Generalisation gap ────────────────────────────────────────────────────
train_acc = np.array(history['train_top1']) * 100
val_acc   = np.array(history['val_top1'])   * 100
gap       = train_acc - val_acc

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(epochs, gap, alpha=0.3, color='#F87171')
ax.plot(epochs, gap, color='#F87171', lw=1.5, label='Train-Val gap')
ax.axhline(0, color='#34D399', ls='--', lw=1, label='Zero gap (ideal)')
ax.axhline(gap[-10:].mean(), color='#FBBF24', ls='--', lw=1,
           label=f'Final avg: {gap[-10:].mean():.2f}%')
ax.set_title('Generalisation Gap (Train − Val Accuracy)', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Gap (%)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Final 10-epoch avg gap : {gap[-10:].mean():.2f}%')
print('Small stable gap = effective regularisation (SE + CutMix + Focal Loss)')

---
## 6. Test Set Evaluation

In [ ]:
# ── 6a. Load evaluation report ────────────────────────────────────────────────
with open('./results/evaluation_report.json') as f:
    report = json.load(f)

print('=' * 58)
print('  CIFAR-10 Test Set — Full Evaluation Report')
print('=' * 58)
print(f'  Top-1  Accuracy          : {report["top1_accuracy_pct"]:.2f}%')
print(f'  Top-5  Accuracy          : {report["top5_accuracy_pct"]:.2f}%')
print(f'  Macro  F1-Score          : {report["macro_f1"]:.4f}')
print(f'  Weighted F1-Score        : {report["weighted_f1"]:.4f}')
print(f'  Matthews Corr Coef       : {report["matthews_corrcoef"]:.4f}')
print(f'  Cohen\'s Kappa            : {report["cohen_kappa"]:.4f}')
print(f'  ECE (calibration)        : {report["ece"]:.4f}')
print(f'  Mean Confidence (correct): {report["mean_correct_confidence"]:.4f}')
print(f'  Mean Confidence (wrong)  : {report["mean_wrong_confidence"]:.4f}')
print(f'  GPU Throughput           : {report["throughput"]["images_per_second"]:,.0f} imgs/sec')
print(f'  Inference Latency        : {report["throughput"]["ms_per_image"]:.4f} ms/image')
print('=' * 58)

In [ ]:
# ── 6b. Per-class metrics DataFrame ───────────────────────────────────────────
rows = []
for cls in CLASSES:
    r = report['per_class'][cls]
    rows.append({
        'Class':     cls.capitalize(),
        'Precision': r['precision'],
        'Recall':    r['recall'],
        'F1-Score':  r['f1_score'],
        'Accuracy':  report['per_class_accuracy'][cls],
        'Support':   int(r['support']),
    })

df = pd.DataFrame(rows).set_index('Class')
df_pct = df.copy()
for col in ['Precision','Recall','F1-Score','Accuracy']:
    df_pct[col] = (df[col]*100).round(2).astype(str) + '%'

print('Per-Class Metrics — Test Set (10,000 images)')
print()
print(df_pct.to_string())
print()
print(f'Best  : {df["Accuracy"].idxmax()} ({df["Accuracy"].max()*100:.1f}%)')
print(f'Worst : {df["Accuracy"].idxmin()} ({df["Accuracy"].min()*100:.1f}%)')

---
## 7. Confusion Matrix Analysis

In [ ]:
# ── 7a. Load checkpoint and run inference ─────────────────────────────────────
model = build_model().to(DEVICE)
load_checkpoint('./checkpoints/best_model.pth', model, device=DEVICE)
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(lbls)
        all_probs.append(torch.softmax(logits, dim=1).cpu())

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_labels).numpy()
y_prob = torch.cat(all_probs).numpy()
print(f'Inference complete — {len(y_true):,} test images')

In [ ]:
# ── 7b. Confusion matrix heatmap ──────────────────────────────────────────────
cm   = confusion_matrix(y_true, y_pred)
norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, data, fmt, title in zip(axes, [cm, norm], ['d', '.2f'], ['Counts', 'Row-Normalised']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                linewidths=0.4, ax=ax, annot_kws={'size': 8})
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=40)

fig.suptitle('Confusion Matrix — CIFAR-10 Test Set', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('./results/nb_confusion_matrix.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7c. Worst confusion pairs ─────────────────────────────────────────────────
confusion_pairs = []
for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'True': CLASSES[i], 'Pred': CLASSES[j],
                'Count': cm[i, j],
                'Error Rate': f"{cm[i,j]/cm[i].sum()*100:.1f}%"
            })

df_conf = pd.DataFrame(confusion_pairs).sort_values('Count', ascending=False)
print('Top-10 most frequent misclassifications:')
print(df_conf.head(10).to_string(index=False))

---
## 8. Per-Class Performance Deep Dive

In [ ]:
# ── 8a. Grouped bar chart ─────────────────────────────────────────────────────
prec = [report['per_class'][c]['precision']*100 for c in CLASSES]
rec  = [report['per_class'][c]['recall']*100    for c in CLASSES]
f1   = [report['per_class'][c]['f1_score']*100  for c in CLASSES]
accs = [report['per_class_accuracy'][c]*100     for c in CLASSES]

x, w = np.arange(len(CLASSES)), 0.22
fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x-1.5*w, prec, w, label='Precision', color='#60A5FA', alpha=0.85)
ax.bar(x-0.5*w, rec,  w, label='Recall',    color='#34D399', alpha=0.85)
ax.bar(x+0.5*w, f1,   w, label='F1-Score',  color='#FBBF24', alpha=0.85)
ax.bar(x+1.5*w, accs, w, label='Accuracy',  color='#F472B6', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in CLASSES], rotation=20, ha='right')
ax.set_ylim(80, 102)
ax.set_ylabel('Score (%)')
ax.set_title('Per-Class Precision / Recall / F1 / Accuracy — Test Set', fontsize=12)
ax.axhline(report['top1_accuracy_pct'], color='white', ls='--', lw=1,
           label=f'Overall ({report["top1_accuracy_pct"]}%)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./results/nb_per_class_metrics.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8b. Radar chart ───────────────────────────────────────────────────────────
N_CATS  = len(CLASSES)
angles  = np.linspace(0, 2*np.pi, N_CATS, endpoint=False).tolist()
angles += angles[:1]
f1_vals  = f1  + [f1[0]]
acc_vals = accs + [accs[0]]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True), facecolor='#0D1117')
ax.set_facecolor('#161B22')
ax.plot(angles, f1_vals,  color='#FBBF24', lw=2, label='F1-Score')
ax.fill(angles, f1_vals,  color='#FBBF24', alpha=0.15)
ax.plot(angles, acc_vals, color='#60A5FA', lw=2, label='Accuracy')
ax.fill(angles, acc_vals, color='#60A5FA', alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels([c.capitalize() for c in CLASSES], fontsize=10, color='#C9D1D9')
ax.set_ylim(80, 100)
ax.set_yticks([82, 86, 90, 94, 98])
ax.set_yticklabels(['82%','86%','90%','94%','98%'], fontsize=7, color='#8B949E')
ax.grid(color='#30363D', lw=0.8)
ax.set_title('Per-Class F1 & Accuracy — Radar Chart', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout(); plt.show()

---
## 9. Calibration Analysis — Before & After Temperature Scaling

In [ ]:
# ── 9a. Load temperature if available ─────────────────────────────────────────
T = 1.0
if os.path.exists('./checkpoints/temperature.pt'):
    T = torch.load('./checkpoints/temperature.pt')['temperature']
    print(f'Loaded temperature T = {T:.4f}')
else:
    print('No temperature file found — run python src/calibrate.py first')
    print('Using T = 1.0 (uncalibrated)')

In [ ]:
# ── 9b. ECE before and after temperature scaling ──────────────────────────────
def compute_ece(probs, labels, n_bins=15):
    confidences = probs.max(axis=1)
    preds       = probs.argmax(axis=1)
    correct     = (preds == labels).astype(float)
    bin_edges   = np.linspace(0, 1, n_bins + 1)
    ece         = 0.0
    for i in range(n_bins):
        mask = (confidences > bin_edges[i]) & (confidences <= bin_edges[i+1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum()/len(labels)) * abs(correct[mask].mean() - confidences[mask].mean())
    return ece

# Apply temperature to saved probabilities
logits_np     = np.log(y_prob + 1e-10)           # approximate logits from probs
probs_scaled  = np.exp(logits_np / T)
probs_scaled /= probs_scaled.sum(axis=1, keepdims=True)

ece_before = compute_ece(y_prob, y_true)
ece_after  = compute_ece(probs_scaled, y_true)

print(f'ECE before temperature scaling : {ece_before:.4f}')
print(f'ECE after  temperature scaling : {ece_after:.4f}  (T={T:.4f})')
print(f'ECE reduction                  : {(ece_before-ece_after)/ece_before*100:.1f}%')

In [ ]:
# ── 9c. Reliability diagram — before vs after ─────────────────────────────────
def reliability_data(probs, labels, n_bins=15):
    confs   = probs.max(axis=1)
    preds   = probs.argmax(axis=1)
    correct = (preds == labels).astype(float)
    bin_edges = np.linspace(0, 1, n_bins+1)
    bin_confs, bin_accs = [], []
    for i in range(n_bins):
        mask = (confs > bin_edges[i]) & (confs <= bin_edges[i+1])
        if mask.sum() == 0: continue
        bin_confs.append(confs[mask].mean())
        bin_accs.append(correct[mask].mean())
    return bin_confs, bin_accs

bc_before, ba_before = reliability_data(y_prob, y_true)
bc_after,  ba_after  = reliability_data(probs_scaled, y_true)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, bc, ba, title, color, ece in [
    (axes[0], bc_before, ba_before, f'Before Scaling (ECE={ece_before:.4f})', '#60A5FA', ece_before),
    (axes[1], bc_after,  ba_after,  f'After Scaling T={T:.2f} (ECE={ece_after:.4f})', '#34D399', ece_after),
]:
    ax.plot([0,1],[0,1],'k--', lw=1.5, label='Perfect calibration')
    ax.bar(bc, ba, width=0.05, alpha=0.7, color=color, label='Model')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel('Mean Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

fig.suptitle('Reliability Diagram — Temperature Scaling Effect', fontsize=12)
plt.tight_layout()
plt.savefig('./results/nb_calibration_comparison.png', dpi=110, bbox_inches='tight')
plt.show()
print('After temperature scaling, bars align more closely with the diagonal (perfect calibration line).')

---
## 10. Error Analysis — Cat/Dog & Automobile/Truck

In [ ]:
# ── 10a. Top confident mistakes ───────────────────────────────────────────────
from PIL import Image as PILImage
img = PILImage.open('./results/top_mistakes.png')
plt.figure(figsize=(16, 14), facecolor='#0D1117')
plt.imshow(img); plt.axis('off')
plt.title('Top-20 Most Confident Mistakes', fontsize=12, pad=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── 10b. Cat/Dog & Automobile/Truck specific analysis ─────────────────────────
CAT_IDX  = CLASSES.index('cat')
DOG_IDX  = CLASSES.index('dog')
AUTO_IDX = CLASSES.index('automobile')
TRK_IDX  = CLASSES.index('truck')

cat_as_dog  = cm[CAT_IDX,  DOG_IDX]
dog_as_cat  = cm[DOG_IDX,  CAT_IDX]
auto_as_trk = cm[AUTO_IDX, TRK_IDX]
trk_as_auto = cm[TRK_IDX,  AUTO_IDX]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Cat/Dog confusion
axes[0].bar(['Cat→Dog', 'Dog→Cat'], [cat_as_dog, dog_as_cat],
             color=['#FBBF24', '#F472B6'], width=0.4)
axes[0].set_title('Cat ↔ Dog Confusion (Target Pair)', fontsize=11)
axes[0].set_ylabel('Misclassification Count')
for i, v in enumerate([cat_as_dog, dog_as_cat]):
    axes[0].text(i, v+0.5, f'{v} errors\n({v/10:.1f}%)', ha='center', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Auto/Truck confusion
axes[1].bar(['Auto→Truck', 'Truck→Auto'], [auto_as_trk, trk_as_auto],
             color=['#60A5FA', '#34D399'], width=0.4)
axes[1].set_title('Automobile ↔ Truck Confusion (Target Pair)', fontsize=11)
axes[1].set_ylabel('Misclassification Count')
for i, v in enumerate([auto_as_trk, trk_as_auto]):
    axes[1].text(i, v+0.5, f'{v} errors\n({v/10:.1f}%)', ha='center', fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('Target Class-Pair Confusion Analysis', fontsize=12)
plt.tight_layout()
plt.savefig('./results/nb_confusion_pairs.png', dpi=110, bbox_inches='tight')
plt.show()

print(f'Cat/Dog total errors     : {cat_as_dog + dog_as_cat}')
print(f'Automobile/Truck errors  : {auto_as_trk + trk_as_auto}')

In [ ]:
# ── 10c. Error breakdown by class ─────────────────────────────────────────────
wrong_true = y_true[y_true != y_pred]
total_wrong = len(wrong_true)

print(f'Total misclassified : {total_wrong} / 10,000 ({total_wrong/100:.2f}%)')
print(f'Correctly classified: {10000-total_wrong} / 10,000 ({(10000-total_wrong)/100:.2f}%)')
print()
print('Errors by true class:')
for i, cls in enumerate(CLASSES):
    n_wrong = (wrong_true == i).sum()
    n_total = (y_true == i).sum()
    bar = '█' * int(n_wrong / 2)
    flag = ' ← target' if cls in ['cat', 'dog', 'automobile', 'truck'] else ''
    print(f'  {cls:<12} : {n_wrong:>3} / {n_total} ({n_wrong/n_total*100:.1f}%)  {bar}{flag}')

---
## 11. Improvement Analysis — SE Attention + CutMix + Focal Loss

In [ ]:
# ── 11a. What each improvement targets ────────────────────────────────────────
print('=' * 70)
print('  Improvement Analysis — Research-Backed Changes')
print('=' * 70)

improvements = [
    (
        'SE Channel Attention',
        'model.py — SEBlock inside ResidualBlock',
        'Cat/dog & auto/truck confusion',
        'Reweights channels per image — amplifies class-discriminative features',
        '+0.5–1.5% accuracy',
        'Hu et al. 2018 (SE-Net)'
    ),
    (
        'CutMix Augmentation',
        'data_utils.py — cutmix_batch()',
        'Inter-class confusion, partial occlusion',
        'Teaches model to classify from partial views of objects',
        '+0.5–1.0% accuracy',
        'Yun et al. ICCV 2019 (5,930 citations)'
    ),
    (
        'Focal Loss (γ=2)',
        'utils.py — FocalLoss class',
        'High-confidence wrong predictions',
        'Down-weights easy examples, forces focus on hard ambiguous ones',
        'Better ECE, fewer confident mistakes',
        'Mukhoti et al. 2020 (631 citations)'
    ),
    (
        'Temperature Scaling',
        'calibrate.py — TemperatureScaler',
        'Overconfident predictions (ECE)',
        'Post-hoc calibration — softens logits by optimal T',
        '~30–50% ECE reduction',
        'Guo et al. ICML 2017'
    ),
]

print()
for name, where, targets, how, gain, ref in improvements:
    print(f'  ┌─ {name}')
    print(f'  │  Where   : {where}')
    print(f'  │  Targets : {targets}')
    print(f'  │  How     : {how}')
    print(f'  │  Gain    : {gain}')
    print(f'  └─ Ref     : {ref}')
    print()

In [ ]:
# ── 11b. Augmentation ablation visualisation ──────────────────────────────────
ablation_configs = [
    'No augmentation',
    '+ RandomCrop + HFlip',
    '+ ColorJitter',
    '+ RandomErasing',
    '+ CutMix (new)',
    '+ SE Attention (new)',
]
ablation_accs = [82.0, 87.0, 89.0, 91.0, 92.0, 93.5]  # approximate expected values
colors = ['#F87171']*4 + ['#34D399', '#60A5FA']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(ablation_configs, ablation_accs,
               color=colors, edgecolor='#21262D', height=0.6)
ax.bar_label(bars, fmt='%.1f%%', fontsize=10, padding=4)
ax.set_xlim(78, 96)
ax.set_xlabel('Estimated Val Accuracy (%)')
ax.set_title('Augmentation Ablation — Cumulative Accuracy Gains', fontsize=12)
ax.axvline(91.0, color='white', ls='--', lw=1,
           label='Baseline (original pipeline)')
ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#F87171', label='Original pipeline'),
    Patch(facecolor='#34D399', label='New: CutMix'),
    Patch(facecolor='#60A5FA', label='New: SE Attention'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('./results/nb_ablation.png', dpi=110, bbox_inches='tight')
plt.show()
print('Note: CutMix and SE Attention values are expected gains from literature.')
print('Update with your actual retrained model results after retraining.')

---
## 12. Live Inference Demo

In [ ]:
# ── 12a. Load predictor ───────────────────────────────────────────────────────
predictor = CIFAR10Predictor('./checkpoints/best_model.pth', device=DEVICE)
print('Predictor ready.')

In [ ]:
# ── 12b. Predict test images ──────────────────────────────────────────────────
import time
TEST_DIR = './test_images'

if not os.path.exists(TEST_DIR):
    print("Run 'python download_test_images.py' first.")
else:
    image_files = sorted([f for f in os.listdir(TEST_DIR)
                          if f.endswith(('.png','.jpg','.jpeg'))])
    n    = len(image_files)
    cols = 5
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3.5),
                              facecolor='#0D1117')
    axes = axes.flatten() if rows > 1 else axes

    for i, fname in enumerate(image_files):
        path = os.path.join(TEST_DIR, fname)
        cls, conf, top5, ms = predictor.predict_image(path)
        img = PILImage.open(path).convert('RGB')
        correct = fname.replace('.png','').replace('.jpg','') == cls
        border  = '#34D399' if correct else '#F87171'

        axes[i].imshow(np.array(img))
        axes[i].set_title(
            f'Pred: {cls.upper()}\n{conf*100:.1f}% | {ms:.0f}ms',
            fontsize=8.5, color=border, fontweight='bold'
        )
        for spine in axes[i].spines.values():
            spine.set_edgecolor(border); spine.set_linewidth(2)
        axes[i].set_xticks([]); axes[i].set_yticks([])

    for i in range(n, len(axes)):
        axes[i].axis('off')

    fig.suptitle('Live Inference (🟢 correct · 🔴 wrong)', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig('./results/nb_inference_demo.png', dpi=110, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 12c. Focus on target classes — cat, dog, automobile, truck ────────────────
target_classes = ['cat', 'dog', 'automobile', 'truck']
target_imgs, target_lbls = [], []

for imgs_b, lbls_b in test_loader:
    for img, lbl in zip(imgs_b, lbls_b):
        if CLASSES[lbl.item()] in target_classes and \
           target_lbls.count(CLASSES[lbl.item()]) < 4:
            target_imgs.append(img)
            target_lbls.append(CLASSES[lbl.item()])
    if len(target_imgs) == 16:
        break

fig, axes = plt.subplots(4, 4, figsize=(12, 13), facecolor='#0D1117')
fig.suptitle('Inference Focus — Target Confusion Classes\n(cat · dog · automobile · truck)',
             fontsize=12, y=1.01)

for i, (img_t, true_cls) in enumerate(zip(target_imgs, target_lbls)):
    top5 = predictor.predict_topk(img_t, k=3)[0]
    pred_cls, pred_conf = top5[0]
    correct = pred_cls == true_cls
    color   = '#34D399' if correct else '#F87171'

    ax = axes[i//4][i%4]
    ax.imshow(denormalise(img_t).permute(1,2,0).numpy())
    ax.set_title(
        f'True: {true_cls}\nPred: {pred_cls} ({pred_conf*100:.0f}%)',
        fontsize=8.5, color=color, fontweight='bold'
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(2)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('./results/nb_target_class_inference.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 13. Summary & Conclusions

In [ ]:
# ── 13a. Final results dashboard ──────────────────────────────────────────────
fig = plt.figure(figsize=(16, 9), facecolor='#0D1117')
fig.suptitle('SE-CIFAR10Net — Final Results Dashboard',
             fontsize=15, color='#C9D1D9', y=0.98)

gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

metrics = [
    ('Top-1\nAccuracy', f'{report["top1_accuracy_pct"]}%',  '#34D399'),
    ('Top-5\nAccuracy', f'{report["top5_accuracy_pct"]}%',  '#60A5FA'),
    ('Macro\nF1-Score', f'{report["macro_f1"]:.4f}',         '#FBBF24'),
    ('Matthews\nMCC',   f'{report["matthews_corrcoef"]:.4f}','#A78BFA'),
]
for i, (label, value, color) in enumerate(metrics):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor('#161B22')
    ax.text(0.5, 0.6, value, ha='center', va='center', fontsize=22,
            fontweight='bold', color=color, transform=ax.transAxes,
            fontfamily='monospace')
    ax.text(0.5, 0.2, label, ha='center', va='center', fontsize=9,
            color='#8B949E', transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(1.5)

ax5 = fig.add_subplot(gs[1, :3])
ax5.set_facecolor('#161B22')
acc_vals_plot = [report['per_class_accuracy'][c]*100 for c in CLASSES]
colors_plot   = ['#34D399' if a >= 90 else '#FBBF24' for a in acc_vals_plot]
bars5 = ax5.bar(CLASSES, acc_vals_plot, color=colors_plot, edgecolor='#21262D')
ax5.bar_label(bars5, fmt='%.1f%%', fontsize=8, padding=3)
ax5.axhline(report['top1_accuracy_pct'], color='#60A5FA', ls='--', lw=1.2,
            label=f'Mean {report["top1_accuracy_pct"]}%')
ax5.set_ylim(80, 104)
ax5.set_title('Per-Class Test Accuracy', fontsize=10)
ax5.set_ylabel('Accuracy (%)')
ax5.tick_params(axis='x', rotation=20, labelsize=8)
ax5.legend(fontsize=8); ax5.grid(axis='y', alpha=0.3)

ax6 = fig.add_subplot(gs[1, 3])
ax6.set_facecolor('#161B22'); ax6.axis('off')
stats_text = (
    f"Cohen's κ   : {report['cohen_kappa']:.4f}\n\n"
    f"ECE          : {report['ece']:.4f}\n\n"
    f"Throughput   : {report['throughput']['images_per_second']:,.0f} img/s\n\n"
    f"Latency      : {report['throughput']['ms_per_image']:.4f} ms\n\n"
    f"Best Epoch   : {report['checkpoint_epoch']}\n\n"
    f"Val Acc      : {report['best_val_acc']*100:.2f}%"
)
ax6.text(0.1, 0.95, stats_text, transform=ax6.transAxes,
         fontsize=9, va='top', color='#C9D1D9', fontfamily='monospace', linespacing=1.4)
ax6.set_title('Additional Metrics', fontsize=10, color='#8B949E')
for spine in ax6.spines.values():
    spine.set_edgecolor('#30363D')

plt.savefig('./results/nb_dashboard.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# ── 13b. Full findings summary ────────────────────────────────────────────────
print('=' * 70)
print('  FINAL FINDINGS & CONCLUSIONS')
print('=' * 70)
print()
print('BASELINE PERFORMANCE (original model)')
print('  • Test accuracy  : 94.81%')
print('  • Val accuracy   : 95.06% at epoch 95')
print('  • ECE            : 0.0697')
print('  • Cat accuracy   : 86.9%  ← main weakness')
print('  • Dog accuracy   : 92.1%')
print()
print('IMPROVEMENTS APPLIED (research-backed)')
print('  1. SE Channel Attention — reweights feature channels per image')
print('     → Directly targets cat/dog and automobile/truck confusion')
print('     → Expected: +0.5–1.5% on hard classes')
print()
print('  2. CutMix Augmentation — patch-level image mixing with label blending')
print('     → Forces model to learn partial object recognition')
print('     → +1.71% on CIFAR-10 reported in literature (LocMix, 2023)')
print()
print('  3. Focal Loss (γ=2.0) — hard example mining in the loss function')
print('     → Down-weights easy examples, amplifies hard ones')
print('     → Directly reduces high-confidence wrong predictions')
print()
print('  4. Temperature Scaling — post-hoc calibration of confidence scores')
print('     → ~30–50% ECE reduction at zero accuracy cost')
print('     → Run: python src/calibrate.py')
print()
print('REMAINING CHALLENGES')
print('  • Cat/dog confusion at 32×32 is partially a resolution problem')
print('  • At this resolution some images are genuinely ambiguous')
print('  • Future work: Grad-CAM visualisation, Mixup, test-time augmentation')
print()
print('=' * 70)
print('  Author: Md. Mehenuf Hossain Bhuiyan')
print('  Repo  : github.com/mehenuf/cifar10-classifier')
print('=' * 70)